# Modul 3 - Hand & Pose Tracking (IMV Laboratory)

**Fokus:** cara kerja landmark dan tracking
**Hands-on:** virtual painter atau kontrol volume dengan jari
**Target:** peserta dapat melacak tangan dan membuat minimal satu interaksi yang dipicu gesture.

Isi notebook:
1. Setup dan fungsi bantu
2. Hand tracking dasar (21 landmark + hitung jari)
3. Virtual Painter
4. Kontrol volume dengan jarak jempol-telunjuk

> **Catatan penting:** notebook ini harus dijalankan **lokal** (Jupyter / VS Code) karena butuh webcam dan jendela `cv2.imshow`.
> Google Colab tidak bisa dipakai untuk webcam langsung. Tekan **`q`** di jendela video untuk menghentikan tiap sel.

## 1. Setup

In [15]:
import sys
print(sys.executable)

c:\Users\Zepunnn\hands-env\Scripts\python.exe


In [16]:
import math
import time

import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

# 0 = webcam default, 1 = webcam lain, atau path video, mis. "demo.mp4"
SOURCE = 0

# Urutan hasil fingers_up(): [jempol, telunjuk, tengah, manis, kelingking]
FINGER_NAMES = ["Jempol", "Telunjuk", "Tengah", "Manis", "Kelingking"]


def open_source(source=SOURCE):
    """Buka webcam (angka) ATAU file video (string path)."""
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(
            f"Tidak bisa membuka sumber video: {source}. "
            "Coba ganti SOURCE (0/1/2) atau pakai file video."
        )
    return cap


def make_hands(max_hands=1, detection_conf=0.7, tracking_conf=0.6):
    return mp_hands.Hands(
        static_image_mode=False,  # False = mode tracking (antar-frame dilacak, lebih cepat)
        max_num_hands=max_hands,
        min_detection_confidence=detection_conf,
        min_tracking_confidence=tracking_conf,
    )


def to_pixels(hand_landmarks, w, h):
    """Landmark MediaPipe berskala 0-1 -> koordinat piksel (x, y)."""
    return [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks.landmark]


def fingers_up(hand_landmarks, label):
    """Return 5 boolean [jempol, telunjuk, tengah, manis, kelingking].

    Asumsi: frame sudah di-flip (mirror) dan telapak menghadap kamera.
    Jari selain jempol: ujung (tip) lebih tinggi dari sendi PIP -> terangkat.
    Jempol: dibandingkan di sumbu X, tergantung tangan kiri/kanan.
    """
    lm = hand_landmarks.landmark
    up = []
    if label == "Right":
        up.append(lm[4].x < lm[3].x)
    else:
        up.append(lm[4].x > lm[3].x)
    for tip in (8, 12, 16, 20):
        up.append(lm[tip].y < lm[tip - 2].y)
    return up


def close_windows():
    """Tutup jendela OpenCV dengan benar (waitKey ekstra dibutuhkan di beberapa OS)."""
    cv2.destroyAllWindows()
    for _ in range(5):
        cv2.waitKey(1)

## 2. Hand Tracking Dasar

MediaPipe Hands mendeteksi **21 landmark** per tangan (0 = pergelangan, 4 = ujung jempol, 8 = ujung telunjuk, 12 = ujung tengah, 16 = ujung manis, 20 = ujung kelingking).

Alur tiap frame: baca frame -> mirror -> BGR ke RGB -> `hands.process()` -> gambar landmark -> hitung jari terangkat.

In [20]:
def run_hand_tracking(source=SOURCE, print_log=False):
    cap = open_source(source)
    hands = make_hands(max_hands=5)
    prev_time = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.flip(frame, 1)  # mirror biar natural seperti cermin
            h, w, _ = frame.shape

            # MediaPipe butuh RGB, OpenCV default BGR
            result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if result.multi_hand_landmarks:
                for hand_lms, handed in zip(result.multi_hand_landmarks, result.multi_handedness):
                    label = handed.classification[0].label  # "Left" / "Right"
                    mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

                    pts = to_pixels(hand_lms, w, h)
                    up = fingers_up(hand_lms, label)

                    cv2.circle(frame, pts[8], 10, (0, 255, 0), cv2.FILLED)  # ujung telunjuk

                    x0, y0 = pts[0]
                    cv2.putText(frame, f"{label}: {sum(up)} jari", (x0 - 40, y0 + 35),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

                    if print_log:
                        names = [n for n, u in zip(FINGER_NAMES, up) if u]
                        print(f"{label} | terangkat: {names or '-'} | telunjuk (px): {pts[8]}")

            now = time.time()
            fps = 1 / (now - prev_time) if prev_time else 0
            prev_time = now
            cv2.putText(frame, f"FPS: {int(fps)}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            cv2.imshow("Hand Tracking (q = keluar)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        hands.close()
        close_windows()


run_hand_tracking()

## 3. Virtual Painter

Menggambar di udara dengan jari telunjuk.

| Gesture | Aksi |
|---|---|
| Telunjuk saja terangkat | **Menggambar** |
| Telunjuk + tengah terangkat | Pindah posisi (pen diangkat) |
| Telapak terbuka (5 jari) ditahan ~1 detik | **Hapus semua** |

Keyboard (fokus di jendela video): `1` biru, `2` hijau, `3` merah, `4` kuning, `e` penghapus, `c` clear, `q` keluar.

In [18]:
PAINTER_COLORS = {
    ord("1"): ((255, 120, 0), "Biru"),
    ord("2"): ((0, 200, 0), "Hijau"),
    ord("3"): ((0, 0, 255), "Merah"),
    ord("4"): ((0, 255, 255), "Kuning"),
}
CLEAR_HOLD_FRAMES = 25  # ~1 detik di 25-30 FPS


def run_virtual_painter(source=SOURCE):
    cap = open_source(source)
    hands = make_hands(max_hands=1)

    canvas = None
    prev_pt = None
    color, color_name = PAINTER_COLORS[ord("1")]
    eraser = False
    clear_counter = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.flip(frame, 1)
            h, w, _ = frame.shape
            if canvas is None:
                canvas = np.zeros_like(frame)

            result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            mode = "Tidak ada tangan"

            if result.multi_hand_landmarks:
                hand_lms = result.multi_hand_landmarks[0]
                label = result.multi_handedness[0].classification[0].label
                mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

                pts = to_pixels(hand_lms, w, h)
                thumb, index, middle, ring, pinky = fingers_up(hand_lms, label)
                tip = pts[8]

                if all([thumb, index, middle, ring, pinky]):
                    # Telapak terbuka -> tahan untuk clear
                    mode = f"Hapus semua... {clear_counter * 100 // CLEAR_HOLD_FRAMES}%"
                    clear_counter += 1
                    prev_pt = None
                    if clear_counter >= CLEAR_HOLD_FRAMES:
                        canvas[:] = 0
                        clear_counter = 0
                else:
                    clear_counter = 0
                    if index and middle and not ring and not pinky:
                        mode = "Pindah (pen diangkat)"
                        prev_pt = None
                        cv2.circle(frame, tip, 12, color, 2)
                    elif index and not middle and not ring and not pinky:
                        mode = "Menggambar" if not eraser else "Menghapus"
                        cv2.circle(frame, tip, 8, color, cv2.FILLED)
                        if prev_pt is not None:
                            if eraser:
                                cv2.line(canvas, prev_pt, tip, (0, 0, 0), 40)
                            else:
                                cv2.line(canvas, prev_pt, tip, color, 8)
                        prev_pt = tip
                    else:
                        mode = "Siap (angkat telunjuk)"
                        prev_pt = None
            else:
                prev_pt = None
                clear_counter = 0

            # Tempelkan canvas ke frame (piksel yang bukan hitam)
            mask = canvas.any(axis=2)
            frame[mask] = canvas[mask]

            cv2.putText(frame, f"Mode: {mode}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame, f"Warna: {'Penghapus' if eraser else color_name}", (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

            cv2.imshow("Virtual Painter (q = keluar)", frame)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("c"):
                canvas[:] = 0
            elif key == ord("e"):
                eraser = not eraser
            elif key in PAINTER_COLORS:
                color, color_name = PAINTER_COLORS[key]
                eraser = False
    finally:
        cap.release()
        hands.close()
        close_windows()


run_virtual_painter()

## 4. Kontrol Volume dengan Jari (alternatif hands-on)

Jarak ujung jempol (landmark 4) ke ujung telunjuk (landmark 8) dipetakan ke volume 0-100%.
Jarak **dinormalisasi** dengan ukuran telapak (landmark 0 ke 5), jadi hasilnya tetap konsisten walau tangan dekat atau jauh dari kamera.
Nilai dihaluskan dengan smoothing (EMA) supaya bar tidak goyang.

Default hanya **simulasi** (bar di layar). Set `SYSTEM_VOLUME = True` untuk mengubah volume Windows sungguhan (butuh `pycaw`).

In [19]:
RATIO_MIN = 0.25   # jari menempel -> 0%   (kalibrasi kalau perlu)
RATIO_MAX = 1.40   # jari terbuka lebar -> 100%
SMOOTH = 0.2       # 0-1, makin kecil makin halus tapi makin lambat
SYSTEM_VOLUME = False  # True = ubah volume Windows (butuh: %pip install pycaw comtypes)


def init_system_volume():
    """Siapkan kontrol volume Windows via pycaw. Return fungsi set(percent) atau None."""
    try:
        from pycaw.pycaw import AudioUtilities
        try:  # pycaw versi baru
            ep = AudioUtilities.GetSpeakers().EndpointVolume
        except AttributeError:  # pycaw versi lama
            from ctypes import POINTER, cast
            from comtypes import CLSCTX_ALL
            from pycaw.pycaw import IAudioEndpointVolume
            dev = AudioUtilities.GetSpeakers()
            iface = dev.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
            ep = cast(iface, POINTER(IAudioEndpointVolume))
        return lambda pct: ep.SetMasterVolumeLevelScalar(pct / 100.0, None)
    except Exception as e:
        print(f"[!] Kontrol volume sistem tidak tersedia ({e}). Lanjut mode simulasi.")
        return None


def run_volume_control(source=SOURCE, system_volume=SYSTEM_VOLUME):
    set_volume = init_system_volume() if system_volume else None
    cap = open_source(source)
    hands = make_hands(max_hands=1)
    vol = 0.0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.flip(frame, 1)
            h, w, _ = frame.shape

            result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

            if result.multi_hand_landmarks:
                hand_lms = result.multi_hand_landmarks[0]
                mp_draw.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)
                pts = to_pixels(hand_lms, w, h)

                thumb_tip, index_tip = pts[4], pts[8]
                palm_size = math.dist(pts[0], pts[5])  # pergelangan -> pangkal telunjuk
                if palm_size > 0:
                    ratio = math.dist(thumb_tip, index_tip) / palm_size
                    target = float(np.interp(ratio, [RATIO_MIN, RATIO_MAX], [0, 100]))
                    vol += (target - vol) * SMOOTH  # smoothing (EMA)
                    if set_volume:
                        set_volume(vol)

                cv2.line(frame, thumb_tip, index_tip, (255, 0, 255), 3)
                cv2.circle(frame, thumb_tip, 10, (255, 0, 255), cv2.FILLED)
                cv2.circle(frame, index_tip, 10, (255, 0, 255), cv2.FILLED)

            # Bar volume
            bar_top, bar_bottom = 100, 400
            fill = int(np.interp(vol, [0, 100], [bar_bottom, bar_top]))
            cv2.rectangle(frame, (30, bar_top), (65, bar_bottom), (255, 255, 255), 2)
            cv2.rectangle(frame, (30, fill), (65, bar_bottom), (0, 255, 0), cv2.FILLED)
            cv2.putText(frame, f"{int(vol)}%", (20, 440), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

            cv2.imshow("Volume Control (q = keluar)", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        cap.release()
        hands.close()
        close_windows()


run_volume_control()

## Troubleshooting

- **Jendela tidak muncul / kamera gagal dibuka:** ganti `SOURCE` (0, 1, 2) atau isi dengan path video, lalu jalankan ulang sel setup.
- **Sel macet dan jendela tidak bisa ditutup:** klik jendela video lalu tekan `q`. Kalau masih macet, *Restart Kernel*.
- **Jempol terbaca terbalik:** pastikan telapak menghadap kamera. Hitungan jempol diasumsikan untuk frame yang sudah di-mirror.
- **Volume tidak berubah:** `SYSTEM_VOLUME = True` hanya jalan di Windows dengan `pycaw` terpasang.